# Telecom Customer Churn — 03. Data Understanding

**Goal:** Understand the dataset structure, variables, categorical values, target distribution, and basic numerical distributions before making cleaning decisions.

> This notebook is intentionally diagnostic. We do not modify the raw dataset here.

## 1. Imports and configuration

We use **Pandas** for data manipulation and **Matplotlib/Seaborn** for basic exploratory plots.

The dataset path is kept in one variable so it can be changed in one place.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Keep the raw dataset path in one place.
RAW_DATA_PATH = Path(r"D:\Data Analytics\Project\2) Dataset\1) Raw\WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Basic plotting configuration.
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load the raw dataset

We load the original file without changing its values or column names.

`raw_df` is our reference copy. It should remain untouched throughout the project.

In [ ]:
raw_df = pd.read_csv(RAW_DATA_PATH)

print(f"Rows: {raw_df.shape[0]:,}")
print(f"Columns: {raw_df.shape[1]}")

## 3. Inspect the first records

`head()` gives us a quick look at the structure and the type of information stored in each row.

In [ ]:
raw_df.head()

## 4. Inspect the dataset shape and columns

- `shape` tells us the number of rows and columns.
- `columns` shows the exact field names we will use later.

In [ ]:
print("Shape:", raw_df.shape)
print("\nColumns:")
print(raw_df.columns.tolist())

## 5. Inspect data types and non-null counts

`info()` is one of the most useful first diagnostics.

It helps us detect:
- numeric columns stored as text,
- missing values,
- the overall memory footprint.

In [ ]:
raw_df.info()

## 6. Count unique values

This gives us a quick view of the cardinality of every field.

For example, fields such as `Contract`, `InternetService`, and `PaymentMethod` should have a small number of meaningful categories.

In [ ]:
raw_df.nunique(dropna=False).sort_values()

## 7. Inspect important categorical fields

We explicitly inspect the main dimensions used in the business questions.

This is better than assuming that the documented categories are exactly what appears in the actual file.

In [ ]:
important_categorical_cols = ["Contract", "InternetService", "PaymentMethod", "Churn"]

for col in important_categorical_cols:
    print(f"\n--- {col} ---")
    print(raw_df[col].unique())

## 8. Distribution of the churn target

We first inspect the counts and then the percentages.

This establishes the baseline churn rate and shows whether the target is imbalanced.

In [ ]:
churn_counts = raw_df["Churn"].value_counts(dropna=False)
churn_pct = raw_df["Churn"].value_counts(normalize=True, dropna=False) * 100

print("Counts:")
print(churn_counts)

print("\nPercentages:")
print(churn_pct.round(2))

## 9. Category distributions

These counts help us understand how customers are distributed across the major business dimensions.

In [ ]:
for col in ["Contract", "InternetService", "PaymentMethod"]:
    print(f"\n=== {col} ===")
    print(raw_df[col].value_counts(dropna=False))

## 10. Churn rate by contract, internet service, and payment method

These are early descriptive checks, not final conclusions.

`normalize="index"` calculates the percentage distribution of Churn within each category.

In [ ]:
for col in ["Contract", "InternetService", "PaymentMethod"]:
    print(f"\n=== Churn by {col} (%) ===")
    table = pd.crosstab(raw_df[col], raw_df["Churn"], normalize="index") * 100
    print(table.round(2))

## 11. Numerical summary

`describe()` provides common descriptive statistics for numeric fields.

We will use this to understand scale, spread, and potential unusual values before the formal quality assessment.

In [ ]:
raw_df.describe()

## 12. Tenure distribution

Tenure is central to our churn analysis because customer age within the telecom relationship may be associated with churn.

This plot is descriptive only; we will investigate tenure groups more systematically later.

In [ ]:
raw_df["tenure"].hist(bins=30, figsize=(8, 5))

plt.xlabel("Tenure (months)")
plt.ylabel("Number of Customers")
plt.title("Customer Tenure Distribution")
plt.show()

## 13. Monthly charges distribution

Monthly charges are important because one of our business questions asks whether high monthly charges are associated with churn.

In [ ]:
raw_df["MonthlyCharges"].hist(bins=30, figsize=(8, 5))

plt.xlabel("Monthly Charges")
plt.ylabel("Number of Customers")
plt.title("Customer Monthly Charges Distribution")
plt.show()

## 14. Total charges distribution — preliminary check

`TotalCharges` is expected to be numeric conceptually, but in the common Telco dataset it may be stored as text because of blank strings.

Therefore, we **do not permanently convert the raw column here**. We create a temporary numeric version only for inspection.

In [ ]:
total_charges_numeric = pd.to_numeric(raw_df["TotalCharges"], errors="coerce")

total_charges_numeric.hist(bins=30, figsize=(8, 5))

plt.xlabel("Total Charges")
plt.ylabel("Number of Customers")
plt.title("Customer Total Charges Distribution")
plt.show()

## 15. Preliminary boxplots

Boxplots help us identify unusually high or low observations.

Important: an outlier is **not automatically an error**. We will investigate these observations in the Data Quality notebook.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.boxplot(x=raw_df["tenure"], ax=axes[0])
axes[0].set_title("Tenure")

sns.boxplot(x=raw_df["MonthlyCharges"], ax=axes[1])
axes[1].set_title("Monthly Charges")

sns.boxplot(x=total_charges_numeric, ax=axes[2])
axes[2].set_title("Total Charges")

plt.tight_layout()
plt.show()